# 單元 9-3 二維陣列初始化與參照共用致命陷阱

- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **對應教材**：第 13 章 list of list 實作二維陣列（第 47 頁）
- **核心目標**：
  1. 親手重現 `[[0] * C] * R` 淺拷貝致命陷阱，徹底理解修改單一元素引發整直行同步變異的根本原因。
  2. 搞懂乘號 `* R` 僅複製物件參照（記憶體位址）之機制，建立深層指標認知。
  3. 掌握正確初始化黃金法則：`[[0] * C for _ in range(R)]`，確保每橫列皆為獨立記憶體個體。
  4. 熟練使用 `id()` 函式探測二維串列各列記憶體門牌號碼，實作自動化獨立性安全驗證。
  5. 掌握 APCS 演算法必備的「布林拜訪矩陣 `[[False]*C]`」與「字元空地圖 `[['.']*C]`」初始化技巧。
  6. 掌握二維陣列的獨立深拷貝技術 `[row[:] for row in grid]`，徹底杜絕資料污染與非預期副作用。

---
### 🧭 單元導航地圖（6 大微型學習階梯）
1. **9.3.1 致命陷阱重現：`[[0] * C] * R` 淺拷貝參照共用慘劇**
2. **9.3.2 正確初始化黃金法則：`[[0] * C for _ in range(R)]`**
3. **9.3.3 記憶體指標探測：用 `id()` 驗證每列記憶體獨立性**
4. **9.3.4 布林標記地圖與字元地圖初始化：`False` 拜訪矩陣與符號地圖**
5. **9.3.5 動態網格生成：依座標規律填入初始數值（如 `r * C + c + 1`）**
6. **9.3.6 二維陣列的獨立深拷貝：`[row[:] for row in grid]`**

### 9.3.1 致命陷阱重現：`[[0] * C] * R` 淺拷貝參照共用慘劇

在第八章我們學過：一維串列可以用 `[0] * 5` 快速造出 `[0, 0, 0, 0, 0]`。許多初學者就會非常自然地推理：**「那如果我要一個 3 列 4 行的二維陣列，是不是寫 `[[0] * 4] * 3` 就好了？」**

**千萬不要這樣寫！這是 Python 程式設計中最著名、也最危險的致命陷阱！**

#### 恐怖實驗：修改一格，整直行跟著發瘋！
```python
bad_grid = [[0] * 4] * 3
bad_grid[0][0] = 7   # 我們明明只改了第 0 列第 0 行！
```
你預期看到的可能是：
```
[7, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
```
但實際印出來卻是：
```
[7, 0, 0, 0]
[7, 0, 0, 0]
[7, 0, 0, 0]   <--- 每一列的第 0 個居然全都變成了 7！
```

#### 為什麼會這樣？
因為內層的 `[0] * 4` 確實造出了一個一維串列。但外層的 `* 3`，**並沒有幫你複製出 3 個新的置物櫃，而是把同一個置物櫃的「鑰匙」複製了 3 份**！
換句話說，`bad_grid[0]`、`bad_grid[1]`、`bad_grid[2]` 根本是**同一個串列在記憶體中的三個分身別名**！修改其中一個，其他人當然立刻同步改變！

In [ ]:
# [2] Code 範例：親手重現淺拷貝參照慘劇

# 錯誤寫法
demo_bad_grid = [[0] * 4] * 3

print("--- 剛建立時看起來很正常 ---")
for demo_row in demo_bad_grid:
    print(demo_row)

# 只修改 (0, 0) 這一格
demo_bad_grid[0][0] = 7

print("\n--- 恐怖！只改了 demo_bad_grid[0][0]，所有橫列同步變成了 7！---")
for demo_row in demo_bad_grid:
    print(demo_row)


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請執行下方使用錯誤語法建立的 trap_grid，
# 將 (1, 2) 號格子修改為 99。
# 請觀察並填入缺空，體驗每一列的第 2 個元素如何同步變異。
# ==========================================
trap_grid = [[0] * 3] * 3

# 修改第 1 列第 2 格
trap_grid[1][___] = 99

print("trap_grid 第 0 列第 2 格數值：", trap_grid[0][2])
print("trap_grid 第 2 列第 2 格數值：", trap_grid[2][2])

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 請故意使用錯誤的乘號語法建立一個 2 列 3 行的 trap_matrix。
# 將 trap_matrix[0][1] 修改為 888。
# 請印出 trap_matrix 的第 1 列。
#
# 【公開測試資料 1】
# 設定：trap_matrix = [[0] * 3] * 2
# 修改 trap_matrix[0][1] = 888
# 預期輸出（驗證第 1 列是否被污染）：
# [0, 888, 0]
#
# 【公開測試資料 2】
# 設定：若為 2 列 2 行且修改 trap_matrix[0][0] = 55
# 預期輸出（第 1 列）：
# [55, 0]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
trap_matrix = [[0] * 3] * 2
trap_matrix[0][1] = 888
print(trap_matrix[1])


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 給定 trap = [[10, 20]] * 4。
# 請不用印出全部，預測 trap[3][0] 在執行 trap[0][0] += 5 後會是多少？
# 請撰寫程式碼驗證你的預測，印出 trap[3][0] 的數值。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.2 正確初始化黃金法則：`[[0] * C for _ in range(R)]`

那麼，在 Python 中到底該如何正確初始化一個 $R$ 列 $C$ 行的全 0 二維陣列呢？

#### 🌟 唯一指定黃金法則：列表生成式
```python
good_grid = [[0] * C for _ in range(R)]
```

#### 為什麼這個寫法是絕對安全的？
1. **內層的 `[0] * C`**：在記憶體中製造出一個全新的、獨立的一維串列。
2. **外層的 `for _ in range(R)`**：每迭代一輪，**就實實在在地重新呼叫一次內層生成動作**！
3. **結果**：外層串列收集到了 $R$ 個**各自擁有獨立記憶體門牌號碼的全新置物櫃**！

現在，當你修改 `good_grid[0][0] = 7` 時，因為 `good_grid[1]` 和 `good_grid[2]` 是完全獨立的物件，它們絕對不會受到任何牽連！這才是最標準、最安全的寫法。

In [ ]:
# [2] Code 範例：正確初始化與獨立性驗證

demo_R, demo_C = 3, 4
# 正確黃金寫法
demo_good_grid = [[0] * demo_C for _ in range(demo_R)]

# 修改單一格子 (0, 0)
demo_good_grid[0][0] = 7

print("--- 成功！只有 (0, 0) 變成 7，其餘各列完全不受影響 ---")
for demo_row in demo_good_grid:
    print(demo_row)


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請使用列表生成式正確建立一個 3 列 2 行的全 0 陣列，
# 並將 (2, 1) 修改為 88。
# ==========================================
R, C = 3, 2

# 補齊正確初始化生成式
grid = [[0] * C for _ in range(___)]

# 修改最後一列最後一格
grid[2][1] = 88

print("第 0 列：", grid[0])
print("第 1 列：", grid[1])
print("第 2 列：", grid[2])

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 給定列數 R = 3，行數 C = 3。
# 請使用正確語法建立全 0 陣列，並將 (1, 1) 改為 999。
# 請分別印出這 3 列，證明其他列的第 1 個元素仍為 0。
#
# 【公開測試資料 1】
# 設定：R = 3, C = 3, grid[1][1] = 999
# 預期輸出：
# [0, 0, 0]
# [0, 999, 0]
# [0, 0, 0]
#
# 【公開測試資料 2】
# 設定：R = 2, C = 2, grid[0][0] = 123
# 預期輸出：
# [123, 0]
# [0, 0]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
R, C = 3, 3
grid = [[0] * C for _ in range(R)]
grid[1][1] = 999
for row in grid:
    print(row)


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 請寫出一段程式碼，動態讀入變數 R 與 C，
# 正確初始化一個 R x C 的全 0 矩陣。
# 接著將四個角落 (0, 0), (0, C-1), (R-1, 0), (R-1, C-1) 全數改為 1。
# 請解包輸出該矩陣。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.3 記憶體指標探測：用 `id()` 驗證每列記憶體獨立性

在科學與工程世界中，「眼見為憑」。我們怎麼能百分之百確信兩者在記憶體中的運作正如我們所言？

#### 記憶體探測器：內建函式 `id()`
在第 8.6 節我們介紹過 `id(obj)`，它會回傳物件在電腦記憶體中的**唯一門牌號碼（記憶體位址）**：
- 如果兩個變數代表同一個實體物件，它們的 `id()` 必然**完全相同**。
- 如果是兩個不同的獨立物件，即使裡面的數字長得一模一樣，它們的 `id()` 也必然**完全不同**！

#### 科學實驗對照：
1. **錯誤寫法 `bad = [[0] * 3] * 3`**：
   - `id(bad[0])` 與 `id(bad[1])`：**完全相同！** 這以無可辯駁的鐵證證明了它們在記憶體中根本就是同一個實體。
2. **正確寫法 `good = [[0] * 3 for _ in range(3)]`**：
   - `id(good[0])` 與 `id(good[1])`：**完全不同！** 證明每一列都是獨立被孵化出來的。

#### 自動化安全檢查技巧：
```python
is_safe = (id(grid[0]) != id(grid[1]))
```
只要此布林值為 `True`，就代表你的二維陣列安全無虞，可以放心修改！

In [ ]:
# [2] Code 範例：記憶體位址科學實測

demo_bad = [[0] * 3] * 3
print("=== 錯誤寫法的每列記憶體位址 ===")
print("id(demo_bad[0])：", id(demo_bad[0]))
print("id(demo_bad[1])：", id(demo_bad[1]))
print("id(demo_bad[2])：", id(demo_bad[2]))
print("第 0 列與第 1 列是同一個物件嗎？", demo_bad[0] is demo_bad[1])

demo_good = [[0] * 3 for _ in range(3)]
print("\n=== 正確寫法的每列記憶體位址 ===")
print("id(demo_good[0])：", id(demo_good[0]))
print("id(demo_good[1])：", id(demo_good[1]))
print("id(demo_good[2])：", id(demo_good[2]))
print("第 0 列與第 1 列是同一個物件嗎？", demo_good[0] is demo_good[1])


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請使用 id() 比對 my_grid 的第 0 列與第 1 列，
# 檢查兩者是否具有不同的記憶體位址，並印出檢查結果。
# ==========================================
my_grid = [[0] * 4 for _ in range(3)]

addr_0 = id(my_grid[0])
addr_1 = ___(my_grid[1])

is_independent = (addr_0 != addr_1)
print("兩列記憶體位址獨立性驗證：", is_independent)

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 給定二維陣列 test_grid。
# 請撰寫邏輯判斷第 0 列與第 1 列是否為同一個物件（使用 is 運算子），
# 若是同一個物件請輸出「警告: 參照共用」；
# 若不是請輸出「安全: 記憶體獨立」。
#
# 【公開測試資料 1】
# 設定：test_grid = [[0]*3 for _ in range(2)]
# 預期輸出：
# 安全: 記憶體獨立
#
# 【公開測試資料 2】
# 設定：test_grid = [[0]*3] * 2
# 預期輸出：
# 警告: 參照共用
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
test_grid = [[0] * 3 for _ in range(2)]
if test_grid[0] is test_grid[1]:
    print("警告: 參照共用")
else:
    print("安全: 記憶體獨立")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 給定一個長度為 R 的二維陣列 g。
# 請寫一個迴圈比對所有相鄰列（0與1, 1與2...），
# 檢查是否每一對相鄰列的 id 都不相等，
# 全部獨立則輸出「全網格安全: True」，否則輸出「全網格安全: False」。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.4 布林標記地圖與字元地圖初始化：`False` 拜訪矩陣與符號地圖

許多初學者在學完二維初始化後，往往只會寫 `[[0] * C for _ in range(R)]`，誤以為二維陣列只能裝數字 0。但在 APCS 的進階題型中，我們最常需要初始化的其實是**「布林標記地圖（Visited Grid）」**與**「符號空白地圖」**！

#### 1. 拜訪標記矩陣（Visited Matrix，APCS 尋路必備）：
- 在迷宮走訪或圖論走訪時，為了防止角色在同一格「無窮迴圈打轉」，我們必須開一個與地圖大小一模一樣的布林矩陣。
- **起始狀態**：全地圖每一格都還沒拜訪過，全部填 `False`！
  ```python
  visited = [[False] * C for _ in range(R)]
  ```
- **走過時更新**：當走到了座標 `(r, c)`，就立刻執行 `visited[r][c] = True`！

#### 2. 符號空白地圖（遊戲與幾何繪圖必備）：
- 畫星號圖案、圍棋盤或踩地雷時，先建立全點點 `'.'` 的空地圖：
  ```python
  grid = [['.'] * C for _ in range(R)]
  ```
- 同樣的道理，內層只要替換預設值（`False`、`'.'`、`'#'`），外層維持 `for _ in range(R)`，就能隨心所欲造出任何型態的乾淨二維畫布！

In [ ]:
# [2] Code 範例：初始化布林拜訪矩陣與字元空地圖

demo_R, demo_C = 3, 3

# 1. 建立布林拜訪矩陣
demo_visited = [[False] * demo_C for _ in range(demo_R)]
print("--- 初始拜訪狀態 ---")
for demo_row in demo_visited:
    print(demo_row)

# 模擬走過起點 (0, 0) 與中心點 (1, 1)
demo_visited[0][0] = True
demo_visited[1][1] = True
print("\n--- 走訪起點與中心後 ---")
for demo_row in demo_visited:
    print(demo_row)

# 2. 建立符號畫布地圖
demo_canvas = [['.'] * 4 for _ in range(2)]
demo_canvas[0][3] = "*"  # 在右上角放一顆星星
print("\n--- 符號地圖 ---")
for demo_row in demo_canvas:
    print("".join(demo_row))


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請補齊下方缺空，初始化一個 2 列 4 行的布林陣列 visited，
# 初始值全部填入 False；並將 (1, 3) 標記為 True。
# ==========================================
R, C = 2, 4

# 初始化全 False
visited = [[___] * C for _ in range(R)]

# 標記右下角為已拜訪
visited[1][3] = ___

print("第 0 列：", visited[0])
print("第 1 列：", visited[1])

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 請建立一個 3 列 3 行的字元陣列 grid，初始全部為 "."。
# 接著請將 (0, 0)、(1, 1)、(2, 2) 均修改為 "O"。
# 最後依序印出這 3 列（每列使用 "".join(row) 緊密輸出）。
#
# 【公開測試資料 1】
# 設定：依題意建立 3x3 地圖並修改對角線為 "O"
# 預期輸出：
# O..
# .O.
# ..O
#
# 【公開測試資料 2】
# 設定：若將四角 (0,0), (0,2), (2,0), (2,2) 改為 "X"
# 預期輸出：
# X.X
# ...
# X.X
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
grid = [['.'] * 3 for _ in range(3)]
grid[0][0] = "O"
grid[1][1] = "O"
grid[2][2] = "O"
for row in grid:
    print("".join(row))


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 請初始化一個 4 列 5 行的布林陣列 visited，初始全為 False。
# 請將最外圍一圈的所有格子（第 0 列、第 R-1 列、第 0 行、第 C-1 行）
# 全部設定為 True（代表邊界已封閉）。
# 請解包印出這個布林網格。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.5 動態網格生成：依座標規律填入初始數值（如 `r * C + c + 1`）

除了全 0、全 `False` 之外，很多題目需要我們建立**「有規律數值」**的二維陣列（例如連續編號 1 到 $R 	imes C$、乘法表、或是黑白相間棋盤格）。

#### 二維座標映射為一維序號的數學公式：
當我們要把一張 $R$ 列 $C$ 行的網格由左至右、由上到下依序編號成 $1, 2, 3, \dots, R \times C$ 時：
- 第 $r$ 列前面已經完整走過了 $r$ 個橫列，每個橫列有 $C$ 個數字，因此前面跳過了 $r \times C$ 個數字。
- 在當前橫列中，又往右走了 $c$ 格。
- **編號公式（從 1 開始）**：
  $$\text{數值} = r \times C + c + 1$$

#### 搭配巢狀生成式動態填值：
```python
numbered_grid = [[r * C + c + 1 for c in range(C)] for r in range(R)]
```
外層 `r` 從 0 到 $R-1$，內層 `c` 從 0 到 $C-1$，每一格的值直接由數學公式算出，一行搞定！

In [ ]:
# [2] Code 範例：規律網格生成

demo_R, demo_C = 3, 4
# 生成 1 到 12 連續編號矩陣
demo_grid = [[r * demo_C + c + 1 for c in range(demo_C)] for r in range(demo_R)]

print("1 到 12 連續編號矩陣：")
for demo_row in demo_grid:
    for demo_val in demo_row:
        print(f"{demo_val:4d}", end="")
    print()


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請填入公式，生成一個 3x3 的棋盤交錯網格：
# 若 (r + c) 是偶數填 0，若是奇數填 1。
# ==========================================
N = 3
checkerboard = [[(r + c) % ___ for c in range(N)] for r in range(N)]

print("黑白棋盤格 (0與1交錯)：")
for row in checkerboard:
    print(*row)

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 請使用列表生成式，建立一個 2 列 3 行的矩陣，
# 每一格的值為其「列索引乘行索引」的乘積：r * c。
# 請依序解包印出每一橫列。
#
# 【公開測試資料 1】
# 設定：R = 2, C = 3
# 預期輸出：
# 0 0 0
# 0 1 2
# （說明：第 0 列 r=0，乘積全為 0；第 1 列 r=1，c 為 0,1,2，乘積為 0,1,2）
#
# 【公開測試資料 2】
# 設定：R = 3, C = 2
# 預期輸出：
# 0 0
# 0 1
# 0 2
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
R, C = 2, 3
grid = [[r * c for c in range(C)] for r in range(R)]
for row in grid:
    print(*row)


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 給定 N = 4，請用列表生成式建立一個 N x N 的「座標和矩陣」，
# 每一格的值為 r + c。
# 請解包印出該矩陣。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.6 二維陣列的獨立深拷貝：`[row[:] for row in grid]`

在第八章我們學過：一維串列只要寫 `b = a.copy()` 或是 `b = a[:]`，就能複製出一個乾淨獨立的副本，修改 `b` 不會影響到 `a`。

**但在二維陣列中，`grid.copy()` 依然會掉入淺拷貝陷阱！**

#### 為什麼二維不能只用 `grid.copy()`？
- `grid.copy()` 確實複製了一個新的外層串列，但外層串列裡面裝著的每一個內層小串列，**依然指向同一個記憶體位址**！
- 結果：修改新矩陣的某格 `new_grid[0][0] = 99`，舊矩陣的 `old_grid[0][0]` 依然會被一起竄改！

#### 二維獨立深拷貝（Deep Copy）純語法解法：
在不載入額外外部模組的前提下，最優雅且高效的複製方式，就是**針對每一列都獨立切片一次**：
```python
new_grid = [row[:] for row in old_grid]
```
- `for row in old_grid`：逐一拿出舊網格中的每一橫列。
- `row[:]`：對該橫列進行完整的獨立切片拷貝！
- 如此一來，新矩陣與舊矩陣從外層到內層，記憶體全部徹底隔離，再也不用擔心資料污染！

In [ ]:
# [2] Code 範例：grid.copy() 淺拷貝陷阱 vs [row[:] for row in grid] 真正獨立深拷貝

demo_original = [[1, 2], [3, 4]]

# 錯誤的複製：grid.copy()
demo_shallow = demo_original.copy()
demo_shallow[0][0] = 999
print("使用 demo_original.copy() 後修改，demo_original[0][0] 慘遭連帶竄改：", demo_original[0][0])

# 重置 demo_original
demo_original = [[1, 2], [3, 4]]

# 正確的二維獨立深拷貝
demo_deep = [row[:] for row in demo_original]
demo_deep[0][0] = 888

print("\n使用 [row[:] for row in demo_original] 後修改：")
print("demo_deep[0][0] 新值：", demo_deep[0][0])
print("demo_original[0][0] 依然安好保持原狀：", demo_original[0][0])


In [ ]:
# ==========================================
# [3] Code 填空引導題
# 任務說明：
# 請補齊切片語法，建立 base_grid 的完全獨立副本 backup_grid。
# ==========================================
base_grid = [[10, 20], [30, 40]]

# 針對每列進行獨立切片
backup_grid = [row[___] for row in base_grid]

# 修改備份矩陣
backup_grid[0][0] = 9999

print("備份矩陣 (0, 0)：", backup_grid[0][0])
print("原始矩陣 (0, 0) 是否維持 10？", base_grid[0][0] == 10)

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 給定二維陣列 src = [[1, 1], [1, 1]]。
# 請使用 [row[:] for row in src] 建立獨立副本 dst，
# 並將 dst[1][1] 改為 0。
# 請印出 src 的第 1 列，驗證其未被修改。
#
# 【公開測試資料 1】
# 設定：src 如上
# 預期輸出：
# [1, 1]
#
# 【公開測試資料 2】
# 設定：若 src = [[5, 6], [7, 8]]，將 dst[0][0] 改為 99
# 預期輸出（src[0]）：
# [5, 6]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
src = [[1, 1], [1, 1]]
dst = [row[:] for row in src]
dst[1][1] = 0
print(src[1])


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 給定二維陣列 maze = [["S", "."], [".", "E"]]。
# 請深拷貝出一份 maze_copy，並在 maze_copy 中將 "S" 與 "E" 都替換為 "*"。
# 請印出原始的 maze，確認起點與終點依舊完好無損。
# （本題為自由挑戰題，無公開測資，請自行實作）
# ==========================================
# 請在下方撰寫你的程式碼：


---
### 🧪【自主學習成效驗收】單元 9-3 一鍵自動評分與身分認證

恭喜你完成了單元 9-3 的所有探索與挑戰！🎉

剛才在各題中建立的數值型態變數、運算式與語法結構，現在都乖乖躺在記憶體置物櫃裡囉。  
點擊下方的 **▶ 播放鍵**，系統將跳出輸入框讓您登記**姓名**與**學號座號**（分開獨立欄位，亦可直接按 Enter 採用自學預設值），並自動啟動 Google 官方帳號身分認證（記錄真實 Email 與 Google 顯示姓名），產出**學習成果成績單**並同步登錄後台試算表！

> 💡 **貼心小叮嚀**：如果評分報告顯示某題未通過，別灰心！只要回到上方對應題目的儲存格修正程式碼，**重新點擊該題播放鍵 ▶ 執行**，再回來重新執行此評分儲存格就可以刷新成績與紀錄囉！

> 🆘 **終極救援方案**：如果重新執行個別題目後，成績還是怪怪的（例如明明寫對了卻還是顯示 ❌），最保險的做法是**整本全部重跑**，一鍵重置記憶體、告別幽靈變數！**Google Colab** 請點上方選單「**執行階段 ➔ 重新啟動工作階段並全部執行**」（快捷鍵 `Ctrl + F9`）；**VS Code** 請於上方工具列依序點選「**Clear All Outputs ➔ Restart ➔ Run All**」。

In [ ]:
#@title 🧪 點擊此處取得單元 9-3 學習成效自動評分報告 { display-mode: "form" }
# ------------------------------------------------------------------------------
# 官方智慧全環境評分載入器（支援：本地開發庫優先 ➔ 公開正式庫雙重備援）
# ------------------------------------------------------------------------------
import os
import sys
import json
import logging
import urllib.request

# 靜音內部 httplib2 警告
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

# 1. 互動式輸入學員姓名與學號（分開獨立欄位，按 Enter 採用預設值）
print("👤【學員身分登記】（自學測試可直接按 Enter 鍵使用預設值）：")
student_name = input("請輸入您的姓名 [預設: 自學冒險者]：").strip() or "自學冒險者"
student_id = input("請輸入學號或座號 [預設: 無座號]：").strip() or "無座號"

# 2. 自動嘗試獲取 Google 帳號官方身分認證（真實 Email 與 Google 暱稱）
google_email = None
google_name = None
try:
    from google.colab import auth
    print("🔐 正在連結 Google 帳號進行身分認證（若跳出授權視窗，請點選允許）...")
    auth.authenticate_user()

    # 方法 1: 使用 Google Drive API v3 (about:get) 抓取真實 displayName 與 emailAddress
    # Colab ADC 預設包含 drive 權限，能 100% 精準獲取 Google 帳號中文顯示名稱（如：葉明昌）
    try:
        import google.auth
        from googleapiclient.discovery import build
        creds, _ = google.auth.default()
        drive_service = build('drive', 'v3', credentials=creds, cache_discovery=False)
        about = drive_service.about().get(fields="user(displayName,emailAddress)").execute()
        u_obj = about.get("user", {})
        google_name = u_obj.get("displayName")
        google_email = u_obj.get("emailAddress")
    except Exception:
        pass

    # 方法 2: 若 Drive API 失敗，嘗試透過 OAuth2 userinfo / tokeninfo 備援
    if not google_email or not google_name:
        try:
            import google.auth
            from google.auth.transport.requests import Request
            credentials, _ = google.auth.default()
            credentials.refresh(Request())
            token = credentials.token

            for ep in [
                f"https://www.googleapis.com/oauth2/v3/tokeninfo?access_token={token}",
                "https://www.googleapis.com/oauth2/v3/userinfo",
                "https://www.googleapis.com/oauth2/v2/userinfo"
            ]:
                try:
                    req = urllib.request.Request(
                        ep,
                        headers={"Authorization": f"Bearer {token}"}
                    )
                    with urllib.request.urlopen(req, timeout=5) as resp:
                        u_data = json.loads(resp.read().decode('utf-8'))
                        if not google_email:
                            google_email = u_data.get("email") or u_data.get("emailAddress")
                        if not google_name:
                            google_name = (
                                u_data.get("name")
                                or u_data.get("given_name")
                                or u_data.get("displayName")
                            )
                        if google_email and google_name:
                            break
                except Exception:
                    continue
        except Exception:
            pass

    # 方法 3: 若只有 email 但 Google API 沒給 name，以 email 前綴作為暱稱回退
    if google_email and not google_name:
        google_name = google_email.split("@")[0]

    if google_email:
        print(f"✅ 身分認證成功！已綁定 Google 帳號：{google_name} ({google_email})")
except Exception:
    # 非 Colab 環境、本機離線測試或略過授權
    pass

# 3. 身分資料獨立注入環境變數（學號與姓名完全分開）
os.environ["STUDENT_NAME"] = student_name
os.environ["STUDENT_ID"] = student_id
if google_email:
    os.environ["GOOGLE_EMAIL"] = google_email
if google_name:
    os.environ["GOOGLE_NAME"] = google_name

# 4. 評分核心腳本載入（本機優先 ➔ 遠端記憶體載入，絕不產生實體暫存檔）
# 找不到來源就換下一條。內容已拿到但執行失敗時，停止改試其他來源。
# 評分腳本放在獨立命名空間執行，成功後也不把名稱寫回筆記本。答案讀執行前的學生存放格快照。
_loaded = False
_load_error = None

# 4.1 優先嘗試讀取開發庫/本機檔案（本機離線開發測試零延遲）
_local_paths = [
    "web/graders/grade_9_3.py",
    "../graders/grade_9_3.py",
    "../../graders/grade_9_3.py",
    "../web/graders/grade_9_3.py",
    "../../web/graders/grade_9_3.py"
]
for _lp in _local_paths:
    if not os.path.exists(_lp):
        continue
    try:
        with open(_lp, "r", encoding="utf-8") as _f:
            _code = _f.read()
        _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
        try:
            exec(_code, _grader_ns)
        finally:
            del _grader_ns
        os.environ["GRADER_SOURCE"] = _lp
        _loaded = True
        break
    except Exception as _exc:
        _load_error = _exc
        break

# 4.2 若在雲端環境（Colab），自動透過公開正式庫雙重備援以純記憶體載入
if not _loaded and _load_error is None:
    _grader_urls = [
        "https://raw.githubusercontent.com/johnnyy-lab/APCS1to3/main/graders/grade_9_3.py",
        "https://johnnyy-lab.github.io/APCS1to3/graders/grade_9_3.py",
        "https://raw.githubusercontent.com/johnnyy-lab/APCS1to3-dev/main/web/graders/grade_9_3.py"
    ]
    for _url in _grader_urls:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=10) as _resp:
                _code = _resp.read().decode("utf-8")
        except Exception:
            continue
        try:
            _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
            try:
                exec(_code, _grader_ns)
            finally:
                del _grader_ns
            os.environ["GRADER_SOURCE"] = _url
            _loaded = True
            break
        except Exception as _exc:
            _load_error = _exc
            break

if not _loaded:
    if _load_error is not None:
        print("❌ 評分程式載入失敗，請稍後重新執行。若仍失敗，請把這個畫面交給老師。")
        if os.environ.get("GRADER_DEBUG", "").strip().lower() in ("1", "true", "yes", "on"):
            _reason = str(_load_error).strip().splitlines()
            _line = _reason[0][:180] if _reason else ""
            if _line:
                print("（開發診斷：" + type(_load_error).__name__ + "：" + _line + "）")
            else:
                print("（開發診斷：" + type(_load_error).__name__ + "）")
    else:
        print("❌ 評分系統連線異常，請確認網路連線狀態（或稍後重新執行）")


## 🎯 學習總結與通關簽到

恭喜你順利攻克了全 Python 課程中最容易引發靈異現象的大魔王地雷：**單元 9-3 二維陣列初始化與參照共用致命陷阱**！

### 本單元核心技能驗收清單：
1. **避開慘劇**：永遠不要寫 `[[0] * C] * R`，警惕乘號複製位址的淺拷貝連環雷。
2. **黃金寫法**：永遠使用 `[[0] * C for _ in range(R)]` 確保每橫列記憶體絕對獨立。
3. **科學驗證**：熟練使用 `id(grid[0]) != id(grid[1])` 進行自動化記憶體防爆檢查。
4. **標記地圖**：掌握 `[[False]*C for _ in range(R)]`（拜訪標記）與 `[['.']*C]`（符號空地圖）初始化。
5. **規律生成**：精通二維座標平鋪序號公式 `r * C + c + 1` 與棋盤交錯運算。
6. **獨立深拷貝**：熟練運用 `[row[:] for row in grid]` 建立百分之百乾淨副本。

---
### 🗺️ 第九章（二維陣列）9 節全景通關地圖：
- 🟢 **9-1 概念與座標存取**（已通關）
- 🟢 **9-2 動態輸入讀取與解包輸出**（已通關）
- 🟢 **9-3 初始化與參照共用致命陷阱**（恭喜通關！）
- ⏳ **9-4 二維網格雙重走訪與行列統計**（下一步：踏入二維演算法的第一步，雙重迴圈時鐘模型！）
- ⏳ **9-5 二維方陣與特殊走訪：對角線與棋盤規律**
- ⏳ **9-6 矩陣幾何操作與逆推還原（APCS b266 專題）**
- ⏳ **9-7 二維網格導航：方向向量與相鄰探測（APCS e287 原型）**
- ⏳ **9-8 網格射線掃描（Raycasting）與連線阻擋（APCS g596 專題）**
- ⏳ **9-9 網格防護與動態模擬：哨兵加框與雙矩陣快照（APCS f313 專題）**

---榮譽授予【二維記憶體拆彈專家徽章】💣！地基已經無比堅固，下一單元我們將正式邁入演算法核心走訪技術：**9-4 二維網格雙重走訪與行列統計**！